# Lab 05 — Split + leakage checks

Мета: детермінований `train/val/test` split + перевірки leakage (exact/near/template/group/time) на `processed_v2`.


## 1) Install deps

In [1]:
!pip -q install -r ../requirements.txt

## 2) Data access (processed_v2 from Lab2)

In [2]:
from pathlib import Path
import json
import re
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

LAB5_ROOT = Path('..').resolve()
LAB2_ROOT = (LAB5_ROOT.parent / 'project_lab2').resolve()
LAB1_ROOT = (LAB5_ROOT.parent / 'project_lab1').resolve()
sys.path.insert(0, str(LAB5_ROOT))

from src.split import make_splits, save_splits

v2_path = LAB2_ROOT / 'data' / 'processed_v2' / 'processed_v2.csv'
print('Reading:', v2_path)
df = pd.read_csv(v2_path)

# Optional group signal from Lab1
lab1_processed = LAB1_ROOT / 'data' / 'processed.csv'
if lab1_processed.exists():
    src_df = pd.read_csv(lab1_processed, usecols=['text_id', 'source'])
    df = df.merge(src_df, on='text_id', how='left')

# Canonical text column for checks
df['text'] = df['text'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print(df.head(2))


Reading: C:\Users\maia1\data\politiekh\masters\nlp\project_lab2\data\processed_v2\processed_v2.csv
Shape: (1000, 5)
Columns: ['text_id', 'text', 'sentences', 'label', 'source']
    text_id                                               text  \
0      9905  Вступив на ІСТ цього року, тепер молюся, щоб п...   
1  10001201  Цифрова держава Повідомлення 123 від 18.04.202...   

                                           sentences  \
0  ["Вступив на ІСТ цього року, тепер молюся, щоб...   
1  ["Цифрова держава Повідомлення 123 від 18.04.2...   

                         label    source  
0  Question / Request for Help  original  
1  Question / Request for Help  original  


## 3) Generate deterministic splits

In [3]:
SEED = 42
STRATEGY = 'stratified_random'

splits = make_splits(
    df,
    strategy=STRATEGY,
    seed=SEED,
    train_size=0.8,
    val_size=0.1,
    test_size=0.1,
    label_col='label',
    id_col='text_id',
)

sample_dir = LAB5_ROOT / 'data' / 'sample'
saved = save_splits(splits, sample_dir)

# Determinism sanity check
splits_again = make_splits(df, strategy=STRATEGY, seed=SEED, label_col='label', id_col='text_id')
deterministic_ok = splits == splits_again

train_ids = set(splits['train_ids'])
val_ids = set(splits['val_ids'])
test_ids = set(splits['test_ids'])

df_train = df[df['text_id'].isin(train_ids)].copy()
df_val = df[df['text_id'].isin(val_ids)].copy()
df_test = df[df['text_id'].isin(test_ids)].copy()

split_sizes = {
    'train': len(df_train),
    'val': len(df_val),
    'test': len(df_test),
    'total': len(df),
}

print('Saved split files:', saved)
print('Split sizes:', split_sizes)
print('Deterministic check:', deterministic_ok)


Saved split files: {'train_ids': WindowsPath('C:/Users/maia1/data/politiekh/masters/nlp/project_lab5/data/sample/splits_train_ids.txt'), 'val_ids': WindowsPath('C:/Users/maia1/data/politiekh/masters/nlp/project_lab5/data/sample/splits_val_ids.txt'), 'test_ids': WindowsPath('C:/Users/maia1/data/politiekh/masters/nlp/project_lab5/data/sample/splits_test_ids.txt')}
Split sizes: {'train': 800, 'val': 100, 'test': 100, 'total': 1000}
Deterministic check: True


## 4) Save split manifest

In [4]:
manifest = {
    'seed': SEED,
    'strategy': STRATEGY,
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'sizes': split_sizes,
    'columns': {
        'id_col': 'text_id',
        'label_col': 'label',
        'group_col': 'source' if 'source' in df.columns else None,
        'time_col': 'date' if 'date' in df.columns else None,
    },
    'deterministic_ok': deterministic_ok,
}

manifest_path = LAB5_ROOT / 'docs' / 'splits_manifest_lab5.json'
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', manifest_path)
print(json.dumps(manifest, ensure_ascii=False, indent=2))


Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab5\docs\splits_manifest_lab5.json
{
  "seed": 42,
  "strategy": "stratified_random",
  "generated_at_utc": "2026-03-16T13:34:34.413981+00:00",
  "sizes": {
    "train": 800,
    "val": 100,
    "test": 100,
    "total": 1000
  },
  "columns": {
    "id_col": "text_id",
    "label_col": "label",
    "group_col": "source",
    "time_col": null
  },
  "deterministic_ok": true
}


def class_pct(x: pd.DataFrame) -> pd.Series:
    return x['label'].value_counts(normalize=True).sort_index() * 100

class_dist = pd.concat(
    [
        class_pct(df_train).rename('train_pct'),
        class_pct(df_val).rename('val_pct'),
        class_pct(df_test).rename('test_pct'),
    ],
    axis=1,
).fillna(0).round(2)

print('Class distribution (%)')
print(class_dist)


def length_stats(x: pd.DataFrame) -> dict:
    words = x['text'].astype(str).apply(lambda t: len(t.split()))
    return {
        'mean_words': float(words.mean()),
        'median_words': float(words.median()),
        'p05_words': float(words.quantile(0.05)),
        'p95_words': float(words.quantile(0.95)),
    }

length_by_split = {
    'train': length_stats(df_train),
    'val': length_stats(df_val),
    'test': length_stats(df_test),
}

print('Length stats by split:')
print(pd.DataFrame(length_by_split).T.round(2))


In [5]:
def class_pct(x: pd.DataFrame) -> pd.Series:
    return x['label'].value_counts(normalize=True).sort_index() * 100

class_dist = pd.concat(
    [
        class_pct(df_train).rename('train_pct'),
        class_pct(df_val).rename('val_pct'),
        class_pct(df_test).rename('test_pct'),
    ],
    axis=1,
).fillna(0).round(2)

print('Class distribution (%)')
print(class_dist)



def length_stats(x: pd.DataFrame) -> dict:
    words = x['text'].astype(str).apply(lambda t: len(t.split()))
    return {
        'mean_words': float(words.mean()),
        'median_words': float(words.median()),
        'p05_words': float(words.quantile(0.05)),
        'p95_words': float(words.quantile(0.95)),
    }

length_by_split = {
    'train': length_stats(df_train),
    'val': length_stats(df_val),
    'test': length_stats(df_test),
}

print('Length stats by split:')
print(pd.DataFrame(length_by_split).T.round(2))


Class distribution (%)
                               train_pct  val_pct  test_pct
label                                                      
Complaint / Dissatisfaction         20.0     20.0      20.0
Gratitude / Positive Feedback       20.0     20.0      20.0
Neutral Comment                     20.0     20.0      20.0
Question / Request for Help         20.0     20.0      20.0
Suggestion / Idea                   20.0     20.0      20.0
Length stats by split:
       mean_words  median_words  p05_words  p95_words
train       22.25          19.0        8.0      44.00
val         24.93          19.0        7.0      46.15
test        22.07          17.5        8.0      46.00


In [6]:
if 'source' in df.columns:
    source_dist = pd.concat(
        [
            df_train['source'].value_counts(normalize=True).rename('train_pct'),
            df_val['source'].value_counts(normalize=True).rename('val_pct'),
            df_test['source'].value_counts(normalize=True).rename('test_pct'),
        ],
        axis=1,
    ).fillna(0).round(4) * 100
    print('Source distribution (%)')
    print(source_dist)
else:
    source_dist = None
    print('No source/group column available.')


Source distribution (%)
          train_pct  val_pct  test_pct
source                                
original      97.25     98.0      99.0
cosmus         2.75      2.0       1.0


## 6) Leakage checks (2.1–2.6)

In [7]:
# 2.1 Exact duplicate leakage by processed text
train_texts = set(df_train['text'].tolist())
val_texts = set(df_val['text'].tolist())
test_texts = set(df_test['text'].tolist())

exact_train_val = len(train_texts & val_texts)
exact_train_test = len(train_texts & test_texts)
exact_val_test = len(val_texts & test_texts)

print('Exact duplicates train∩val =', exact_train_val)
print('Exact duplicates train∩test =', exact_train_test)
print('Exact duplicates val∩test =', exact_val_test)


Exact duplicates train∩val = 2
Exact duplicates train∩test = 0
Exact duplicates val∩test = 0


In [8]:
# 2.2 Near-duplicate leakage (TF-IDF + cosine)
def near_duplicate_pairs(left_df: pd.DataFrame, right_df: pd.DataFrame, threshold: float = 0.95) -> pd.DataFrame:
    left_ids = left_df['text_id'].tolist()
    right_ids = right_df['text_id'].tolist()
    left_text = left_df['text'].astype(str).tolist()
    right_text = right_df['text'].astype(str).tolist()

    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    X = vec.fit_transform(left_text + right_text)
    X_left = X[: len(left_text)]
    X_right = X[len(left_text) :]

    sim = cosine_similarity(X_left, X_right)
    hits = np.argwhere(sim >= threshold)

    rows = []
    for i, j in hits:
        rows.append(
            {
                'left_id': int(left_ids[i]),
                'right_id': int(right_ids[j]),
                'score': float(sim[i, j]),
                'left_text': left_text[i],
                'right_text': right_text[j],
            }
        )

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values('score', ascending=False).reset_index(drop=True)
    return out

near_train_test = near_duplicate_pairs(df_train, df_test, threshold=0.95)
near_train_val = near_duplicate_pairs(df_train, df_val, threshold=0.95)

print('Near-duplicates train vs test:', len(near_train_test))
print('Near-duplicates train vs val:', len(near_train_val))

near_examples = pd.concat([near_train_test.head(3), near_train_val.head(2)], ignore_index=True)
near_examples = near_examples.head(5)
if near_examples.empty:
    print('No suspicious near-duplicate pairs at threshold 0.95')
else:
    print(near_examples[['left_id', 'right_id', 'score']])


Near-duplicates train vs test: 1
Near-duplicates train vs val: 2
   left_id  right_id  score
0    12979     13101    1.0
1    12886     13195    1.0
2    15892     15885    1.0


In [9]:
# 2.3 Template / metadata leakage checks
pattern = re.compile(r'(category\s*:|label\s*=|class\s*=|topic\s*=)', re.IGNORECASE)
class_strings = sorted({str(x).lower() for x in df['label'].dropna().unique()})


def has_template_leak(text: str) -> bool:
    t = str(text)
    if pattern.search(t):
        return True
    tl = t.lower()
    return any(cls in tl for cls in class_strings)

bad_template = df[df['text'].apply(has_template_leak)].copy()
print('Template/metadata leakage rows:', len(bad_template))
print(bad_template[['text_id', 'text']].head(10))


Template/metadata leakage rows:

 0


Empty DataFrame
Columns: [text_id, text]
Index: []


In [10]:
# 2.4 Group leakage (if groups exist)
if 'source' in df.columns:
    g_train = set(df_train['source'].dropna().astype(str).unique())
    g_val = set(df_val['source'].dropna().astype(str).unique())
    g_test = set(df_test['source'].dropna().astype(str).unique())

    group_overlap = {
        'train_val': len(g_train & g_val),
        'train_test': len(g_train & g_test),
        'val_test': len(g_val & g_test),
    }
    print('Group overlap counts:', group_overlap)
else:
    group_overlap = None
    print('No group column available, group leakage check skipped.')

# 2.5 Time leakage (if date exists)
if 'date' in df.columns:
    dtrain = pd.to_datetime(df_train['date'], errors='coerce')
    dval = pd.to_datetime(df_val['date'], errors='coerce')
    dtest = pd.to_datetime(df_test['date'], errors='coerce')
    time_table = {
        'train_min': str(dtrain.min()), 'train_max': str(dtrain.max()),
        'val_min': str(dval.min()), 'val_max': str(dval.max()),
        'test_min': str(dtest.min()), 'test_max': str(dtest.max()),
    }
    time_leak_ok = (dtrain.max() <= dtest.min()) if (dtrain.notna().any() and dtest.notna().any()) else None
    print('Time table:', time_table)
    print('max(train)<=min(test):', time_leak_ok)
else:
    time_table = None
    time_leak_ok = None
    print('No date column available, time leakage check skipped.')


Group overlap counts: {'train_val': 2, 'train_test': 2, 'val_test': 2}
No date column available, time leakage check skipped.


In [11]:
# 2.6 Fit-only-on-train discipline check (for ML)
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ('clf', LinearSVC()),
])

pipe.fit(df_train['text'].astype(str), df_train['label'].astype(str))
val_pred = pipe.predict(df_val['text'].astype(str))
test_pred = pipe.predict(df_test['text'].astype(str))

fit_only_train_ok = True
print('Pipeline fitted on train only: OK')
print('val_pred shape:', val_pred.shape, '| test_pred shape:', test_pred.shape)


Pipeline fitted on train only: OK
val_pred shape: (100,) | test_pred shape: (100,)


## 7) Generate risk report + audit summary + dataset card

In [12]:
report_path = LAB5_ROOT / 'docs' / 'leakage_risk_report_lab5.md'
audit_path = LAB5_ROOT / 'docs' / 'audit_summary_lab5.md'
card_path = LAB5_ROOT / 'docs' / 'dataset_card.md'

near_tt_n = len(near_train_test)
near_tv_n = len(near_train_val)
near_examples_short = []
if not near_examples.empty:
    for _, r in near_examples.iterrows():
        near_examples_short.append(f"- {int(r['left_id'])} vs {int(r['right_id'])} (cos={r['score']:.4f})")

strategy_text = [
    "Обрана стратегія: stratified random split 80/10/10 з фіксованим seed=42.",
    "Для нашого датасету (класифікація з 5 класами, N=1000) це дає стабільні частки класів у train/val/test.",
    "Ця стратегія запобігає перекосам класів, які могли б штучно покращити або погіршити метрики.",
    "Ми явно перевірили duplicate leakage між сплітами, щоб однакові тексти не потрапляли у train і test одночасно.",
    "Додатково перевірено near-duplicates через TF-IDF + cosine similarity (threshold=0.95).",
    "Пошук template leakage показує, чи немає службових підказок на кшталт label=/class=/topic= у тексті.",
    "Дисципліна fit-only-on-train зафіксована через sklearn Pipeline, де fit виконується тільки на train.",
]

report_lines = []
report_lines.append('# Leakage risk report — Lab5')
report_lines.append('')
report_lines.append('## 1) Стратегія split (яка і чому)')
report_lines.extend([f'- {x}' for x in strategy_text])
report_lines.append('')
report_lines.append('## 2) Статистика сплітів (розміри, баланс класів/джерел)')
report_lines.append(f"- Sizes: train={split_sizes['train']}, val={split_sizes['val']}, test={split_sizes['test']}, total={split_sizes['total']}")
for cls in class_dist.index.tolist():
    row = class_dist.loc[cls]
    report_lines.append(f"- {cls}: train={row['train_pct']:.2f}%, val={row['val_pct']:.2f}%, test={row['test_pct']:.2f}%")
if source_dist is not None:
    for src in source_dist.index.tolist():
        row = source_dist.loc[src]
        report_lines.append(f"- source={src}: train={row['train_pct']:.2f}%, val={row['val_pct']:.2f}%, test={row['test_pct']:.2f}%")
report_lines.append('')
report_lines.append('## 3) Leakage checks results')
report_lines.append(f'- exact duplicates train∩test = {exact_train_test}')
report_lines.append(f'- exact duplicates train∩val = {exact_train_val}')
report_lines.append(f'- exact duplicates val∩test = {exact_val_test}')
report_lines.append(f'- near-duplicates train vs test (>=0.95): {near_tt_n}')
report_lines.append(f'- near-duplicates train vs val (>=0.95): {near_tv_n}')
report_lines.append(f'- template leakage rows: {len(bad_template)}')
if group_overlap is not None:
    report_lines.append(f"- group overlap train/val={group_overlap['train_val']}, train/test={group_overlap['train_test']}, val/test={group_overlap['val_test']}")
else:
    report_lines.append('- group leakage: N/A (немає релевантної group-колонки)')
if time_table is not None:
    report_lines.append(f"- time leakage check: max(train)<=min(test) => {time_leak_ok}")
else:
    report_lines.append('- time leakage: N/A (немає date-колонки)')
report_lines.append(f'- fit only on train: {fit_only_train_ok}')
report_lines.append('')
report_lines.append('### 5 прикладів near-duplicate пар')
if near_examples_short:
    report_lines.extend(near_examples_short[:5])
else:
    report_lines.append('- Немає пар вище порогу 0.95.')
report_lines.append('')
report_lines.append('## 4) Ризики, що залишились')
report_lines.append('- Можливі семантично близькі пари з cosine < 0.95 (не покриті цим порогом).')
report_lines.append('- Group leakage не може бути повністю оцінений без author/user/thread ідентифікаторів.')
report_lines.append('- У текстах можуть лишатися доменні шаблони, які не збігаються з regex-патернами template leakage.')
report_lines.append('')
report_lines.append('## 5) Що зробимо далі')
report_lines.append('- Додати MinHash/SimHash для більш чутливого near-duplicate аналізу.')
report_lines.append("- Якщо з\'явиться user/thread/date, перейти на group/time-based split і повторити checks.")
report_lines.append('- Використовувати цей split manifest як фіксовану основу для Lab7-Lab8 метрик.')

report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

# Short audit summary
audit_lines = []
audit_lines.append('# Audit summary — Lab5')
audit_lines.append('')
audit_lines.append(f"- strategy={STRATEGY}, seed={SEED}, split=train/val/test={split_sizes['train']}/{split_sizes['val']}/{split_sizes['test']}")
audit_lines.append(f'- exact_dup: train∩test={exact_train_test}, train∩val={exact_train_val}, val∩test={exact_val_test}')
audit_lines.append(f'- near_dup(>=0.95): train-test={near_tt_n}, train-val={near_tv_n}')
audit_lines.append(f'- template_leak_rows={len(bad_template)}')
audit_lines.append(f'- fit_only_train={fit_only_train_ok}')
audit_lines.append(f'- deterministic_split={deterministic_ok}')
audit_path.write_text('\n'.join(audit_lines) + '\n', encoding='utf-8')

# Dataset card update (Splits & leakage section)
card_lines = []
card_lines.append('# Dataset Card — Lab5 update')
card_lines.append('')
card_lines.append('## Splits & leakage')
card_lines.append(f'- Split strategy: {STRATEGY} (seed={SEED}), sizes train/val/test = {split_sizes["train"]}/{split_sizes["val"]}/{split_sizes["test"]}.')
card_lines.append(f'- Exact duplicate leakage: train∩test={exact_train_test}, train∩val={exact_train_val}, val∩test={exact_val_test}.')
card_lines.append(f'- Near-duplicate leakage (cos>=0.95): train-test={near_tt_n}, train-val={near_tv_n}.')
card_lines.append(f'- Template leakage rows detected: {len(bad_template)}; fit-only-on-train discipline: {fit_only_train_ok}.')
card_path.write_text('\n'.join(card_lines) + '\n', encoding='utf-8')

print('Saved:', report_path)
print('Saved:', audit_path)
print('Saved:', card_path)


Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab5\docs\leakage_risk_report_lab5.md
Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab5\docs\audit_summary_lab5.md
Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab5\docs\dataset_card.md


In [13]:
print('Lab5 artifacts ready:')
print('-', LAB5_ROOT / 'src' / 'split.py')
print('-', LAB5_ROOT / 'docs' / 'splits_manifest_lab5.json')
print('-', LAB5_ROOT / 'docs' / 'leakage_risk_report_lab5.md')
print('-', LAB5_ROOT / 'docs' / 'audit_summary_lab5.md')
print('-', LAB5_ROOT / 'docs' / 'dataset_card.md')
print('-', LAB5_ROOT / 'data' / 'sample' / 'splits_train_ids.txt')
print('-', LAB5_ROOT / 'data' / 'sample' / 'splits_val_ids.txt')
print('-', LAB5_ROOT / 'data' / 'sample' / 'splits_test_ids.txt')


Lab5 artifacts ready:
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab5\src\split.py
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab5\docs\splits_manifest_lab5.json
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab5\docs\leakage_risk_report_lab5.md
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab5\docs\audit_summary_lab5.md
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab5\docs\dataset_card.md
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab5\data\sample\splits_train_ids.txt
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab5\data\sample\splits_val_ids.txt
- C:\Users\maia1\data\politiekh\masters\nlp\project_lab5\data\sample\splits_test_ids.txt
